In [ ]:
# =============================================================================
# AML NPS DASHBOARD  |  LEGACY definition of IC eligible / ineligible
#
# OBJECTIVE : Build the AML 1L new patient start dataset for the NPS dashboard -
#             attribute each 1L regimen to its backbone product and initiating HCP,
#             attach account type and HCP segments, then index the observed SHA share
#             against the reported (Ipsos) share and roll it up by month and account.
#
# DEFINITION USED HERE
#   Patient intensity comes from the regimen patient-cohort table, and the backbone is
#   chosen in pandas by ranking the regimen products through two hardcoded hierarchy
#   lists - one for IC eligible patients, one for IC ineligible. The new-definition
#   notebook instead reads intensity and backbone from the age-and-product table.
#
# GRAIN
#   One row per patient and LINE OF THERAPY (LOT). LOT is carried end to end, so the
#   NPS summary is reported for 1L, 2L, 3L and beyond rather than 1L only.
#
# USER INPUTS (confirm before every run)
#   _UPDATED_BUSINESS_RULES suffixes hardcoded output and input table names
#   product hierarchy lists          two hardcoded orderings drive backbone selection
#   Ipsos repository table           must be refreshed each month before this runs
#   reporting floor 2019-01-01       index and rollup start month
#   export file names                ..._EXTRACT.xlsx, index_map_Jan26.csv, index_map_acct_Jan26.csv
#
# SOURCE TABLES
#   ...PATIENT_INTENSITY_BASED_ON_AGE_N_PRODUCT / ...PATS_LOT_GROUPING_TBL   1L regimens
#   abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw   AML Tx claims with prescriber
#   ABV_DDS_SYND.CUSTOMER_TBL                               NPI to Abbott customer id
#   ABV_ADS_SYND.DIM_ACCOUNT_AFFILIATIONS_TBL, ABV_MA360_MHCDM.MHCD_RLTN_ALL_ORG_CUSTOMR_WKLY_TBL,
#   ...heme_account, ...MABI_AML_CHILD_ACCOUNT_DECILE       Account type and decile
#   ...NPI_ABOVE_BRAND_SEGMENT / ...NPI_GNE_BRAND_SEGMENT_BKP   HCP segments
#   ...MABI_AML_NPS_DASHBOARD_REPOSITORY                    Reported NPS, refreshed monthly
#
# OUTPUT
#   ...REG_NPI_ACI_MAPPING_UPDATED_BUSINESS_RULES     regimen to initiating NPI / account
#   ...REG_HCP_INFO_MAPPING_UPDATED_BUSINESS_RULES    plus HCP segments and account type
#   ...1L_IC_INELIG_PBI_DATA_UPDATED_BUSINESS_RULES   Power BI dataset (final output)
#   ...IPSOS_INDEX_REF_UPDATED_BUSINESS_RULES         reported vs observed NPS share
#   ...1L_IC_INELIG_FINAL_DATA_MONTH_ROLLUP_UPDATED_BUSINESS_RULES   NPS by month/account
#   ...LOT_MONTH_NPS_SUMMARY_UPDATED_BUSINESS_RULES    NPS volume and share by LOT / month
# =============================================================================


#   AML NPS Dashboard - LEGACY Definition of IC Eligible / Ineligible
Backbone is selected in pandas from two hardcoded product hierarchies.

In [ ]:
import findspark 
import pandas as pd
findspark.init ()
import pyspark 
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .config('spark.executor.instances','15')\
		.config('spark.driver.memory','4g')\
		.config('spark.executor.memory','12g')\
		.config('spark.executor.cores','5')\
        .config('spark.executor.memoryOverhead','2g')\
        .appName("AML")\
        .master("yarn")\
        .getOrCreate()

In [ ]:
from pyspark.sql.functions import year, month, col

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
spark.sql("set spark.sql.legacy.timeParserPolicy=LEGACY")
spark.sql("set spark.sql.parquet.int96RebaseModeInRead=LEGACY")
spark.sql("set spark.sql.legacy.createHiveTableByDefault=FALSE")

# GLOBAL VARIABLES

In [ ]:
# USER INPUT REQUIRED: '_UPDATED_BUSINESS_RULES' table suffixes - update per refresh
cws = 'Z_ABV_CWS_MABI_ONC_ANALYTICS'

# FROM REGIMEN SETUP
            
updated_tx_tbl = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2_SATEEK'.format(cws=cws)
ic_inelig_final = '{cws}.MABI_AML_IC_INELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2_SATEEK'.format(cws=cws)
elig_flags_final='{cws}.MABI_AML_LOT_PAT_ELIG_FLAGS_FINAL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

# NPS DASH
reg_npi_aci='{cws}.MABI_AML_NPS_DASHBOARD_REG_NPI_ACI_MAPPING_UPDATED_BUSINESS_RULES_v2_SATEEK'.format(cws=cws)
npi_hcp_seg_map='{cws}.MABI_AML_NPS_DASHBOARD_REG_HCP_INFO_MAPPING_UPDATED_BUSINESS_RULES_v2_SATEEK'.format(cws=cws)
dash_tbl_final='{cws}.MABI_AML_NPS_DASHBOARD_1L_IC_INELIG_PBI_DATA_UPDATED_BUSINESS_RULES_v2_SATEEK'.format(cws=cws)
acct_type_grp_tbl = '{cws}.MABI_AML_NPS_DASHBOARD_1L_IC_INELIG_ACCT_TYPE_GRP_UPDATED_BUSINESS_RULES_v2_SATEEK'.format(cws=cws)

# SUMMARY BY LOT
nps_lot_month_summary='{cws}.MABI_AML_NPS_DASHBOARD_LOT_MONTH_NPS_SUMMARY_UPDATED_BUSINESS_RULES_v2_SATEEK'.format(cws=cws)

# INDEXING
ipsos_index_tbl='{cws}.MABI_AML_NPS_DASHBOARD_IPSOS_INDEX_REF_UPDATED_BUSINESS_RULES_v2'.format(cws=cws)
nps_dash_acct_month_rollup='{cws}.MABI_AML_NPS_DASHBOARD_1L_IC_INELIG_FINAL_DATA_MONTH_ROLLUP_UPDATED_BUSINESS_RULES_v2'.format(cws=cws)

# RELTIO AFF
reltio_tbl = 'ABV_MA360_MHCDM.MHCD_RLTN_ALL_ORG_CUSTOMR_WKLY_TBL'

# STATIC MAPPING TABLES
heme_account='{cws}.heme_account'.format(cws=cws) # UPLOADED BY COE
above_brand='{cws}.MABI_AML_NPS_DASHBOARD_NPI_ABOVE_BRAND_SEGMENT'.format(cws=cws)
#gne_brand='{cws}.MABI_AML_NPS_DASHBOARD_NPI_GNE_BRAND_SEGMENT_v2'.format(cws=cws)
# bkp with 15k AML Segmented HCPs
gne_brand='{cws}.MABI_AML_NPS_DASHBOARD_NPI_GNE_BRAND_SEGMENT_BKP'.format(cws=cws)
# Dependency: the reported NPS repository must be reloaded before this notebook runs
ipsos_repo_tbl='{cws}.MABI_AML_NPS_DASHBOARD_REPOSITORY'.format(cws=cws) # UPDATED EACH MONTH

In [ ]:
import sys
sys.path.append('../../config')
from load_config import load_config
import intensity_definition as idef

dcfg = load_config('nps_dashboard_legacy.yaml')
INTENSITY_DEFINITION = idef.definition(dcfg)     # 'legacy' (default) or 'new'
DEF_TAG = idef.tag(dcfg)
print('IC eligible / ineligible definition:', idef.label(dcfg))

# Inputs, as before for legacy runs.
lot_grouping_in = '{cws}.MABI_AML_PATS_LOT_GROUPING_TBL_BUSINESS_RULE_CHANGE_VAL_v2_SATEEK'.format(cws=cws)
cohort_in = '{cws}.MABI_AML_REGIMEN_REVAMP_PATIENT_COHORT_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
flags_in = '{cws}.MABI_AML_LOT_PAT_ELIG_FLAGS_FINAL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
if INTENSITY_DEFINITION == 'new':
    # New-definition base layer (LoT and eligibility run with intensity_definition: new). The cohort
    # PATIENT_INTENSITY_GROUP is the new class; the line backbone below uses it to pick the rank list.
    lot_cfg = load_config('intensity_lot.yaml')
    lot_grouping_in = lot_cfg['tables']['lot_regimen_group'] + DEF_TAG
    cohort_in = lot_cfg['tables']['patient_group'] + DEF_TAG
    flags_in = elig_flags_final + DEF_TAG
    elig_flags_final = flags_in
    # Dashboard outputs get the tag so the legacy dashboard tables are not overwritten.
    for _v in ('reg_npi_aci', 'npi_hcp_seg_map', 'dash_tbl_final', 'acct_type_grp_tbl',
               'nps_lot_month_summary', 'ipsos_index_tbl', 'nps_dash_acct_month_rollup'):
        globals()[_v] = globals()[_v] + DEF_TAG
print('Inputs:', lot_grouping_in, cohort_in, flags_in)

# MAX MONTH CALCULATION

####   Reporting Date

In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta  

curr_dt = datetime.today()
data_dt = curr_dt + relativedelta(months=-3)
max_data_dt = data_dt.strftime("%Y-%m-%d")
month = data_dt.strftime("%b%y").upper()

In [ ]:
print("Today's date: ", curr_dt)
print("Date for data ending (for backup): ", max_data_dt)
print("Month for data ending (for backup): ", month)

# GLOBAL VARIABLE FOR BACKUP TABLES

In [ ]:
# Backup table names and the backup cells below are commented out in this version
# cws = 'Z_ABV_CWS_MABI_ONC_ANALYTICS'

# # NPS DASH
# reg_npi_aci_bkp='{cws}.MABI_AML_NPS_DASHBOARD_REG_NPI_ACI_MAPPING_BKP_{month}'.format(cws=cws, month=month)
# npi_hcp_seg_map_bkp='{cws}.MABI_AML_NPS_DASHBOARD_REG_HCP_INFO_MAPPING_BKP_{month}'.format(cws=cws, month=month)
# dash_tbl_final_bkp='{cws}.MABI_AML_NPS_DASHBOARD_1L_IC_INELIG_PBI_DATA_BKP_{month}'.format(cws=cws, month=month)

# # INDEXING
# ipsos_index_tbl_bkp='{cws}.MABI_AML_NPS_DASHBOARD_IPSOS_INDEX_REF_BKP_{month}'.format(cws=cws, month=month)
# nps_dash_acct_month_rollup_bkp='{cws}.MABI_AML_NPS_DASHBOARD_1L_IC_INELIG_FINAL_DATA_MONTH_ROLLUP_BKP_{month}'.format(cws=cws, month=month)

# BACKUP TABLES

In [ ]:
# spark.sql('''DROP TABLE IF EXISTS {reg_npi_aci_bkp}'''.format(reg_npi_aci_bkp=reg_npi_aci_bkp))
# # spark.sql('''CREATE TABLE {reg_npi_aci_bkp} AS (SELECT * FROM {reg_npi_aci})'''.format(reg_npi_aci_bkp=reg_npi_aci_bkp, reg_npi_aci=reg_npi_aci))

In [ ]:
# spark.sql('''SELECT COUNT(*) FROM {reg_npi_aci_bkp}'''.format(reg_npi_aci_bkp=reg_npi_aci_bkp)).toPandas()

In [ ]:
# spark.sql('''DROP TABLE IF EXISTS {npi_hcp_seg_map_bkp}'''.format(npi_hcp_seg_map_bkp=npi_hcp_seg_map_bkp))
# spark.sql('''CREATE TABLE {npi_hcp_seg_map_bkp} AS (SELECT * FROM {npi_hcp_seg_map})'''.format(npi_hcp_seg_map_bkp=npi_hcp_seg_map_bkp, npi_hcp_seg_map=npi_hcp_seg_map))

In [ ]:
# spark.sql('''SELECT COUNT(*) FROM {npi_hcp_seg_map_bkp}'''.format(npi_hcp_seg_map_bkp=npi_hcp_seg_map_bkp)).toPandas()

In [ ]:
# spark.sql('''DROP TABLE IF EXISTS {dash_tbl_final_bkp}'''.format(dash_tbl_final_bkp=dash_tbl_final_bkp))
# spark.sql('''CREATE TABLE {dash_tbl_final_bkp} AS (SELECT * FROM {dash_tbl_final})'''.format(dash_tbl_final_bkp=dash_tbl_final_bkp, dash_tbl_final=dash_tbl_final))

In [ ]:
# spark.sql('''SELECT COUNT(*) FROM {dash_tbl_final_bkp}'''.format(dash_tbl_final_bkp=dash_tbl_final_bkp)).toPandas()

In [ ]:
# spark.sql('''DROP TABLE IF EXISTS {ipsos_index_tbl_bkp}'''.format(ipsos_index_tbl_bkp=ipsos_index_tbl_bkp))
# spark.sql('''CREATE TABLE {ipsos_index_tbl_bkp} AS (SELECT * FROM {ipsos_index_tbl})'''.format(ipsos_index_tbl_bkp=ipsos_index_tbl_bkp, ipsos_index_tbl=ipsos_index_tbl))

In [ ]:
# spark.sql('''SELECT COUNT(*) FROM {ipsos_index_tbl_bkp}'''.format(ipsos_index_tbl_bkp=ipsos_index_tbl_bkp)).toPandas()

In [ ]:
# spark.sql('''DROP TABLE IF EXISTS {nps_dash_acct_month_rollup_bkp}'''.format(nps_dash_acct_month_rollup_bkp=nps_dash_acct_month_rollup_bkp))
# spark.sql('''CREATE TABLE {nps_dash_acct_month_rollup_bkp} AS (SELECT * FROM {nps_dash_acct_month_rollup})'''.format(nps_dash_acct_month_rollup_bkp=nps_dash_acct_month_rollup_bkp, nps_dash_acct_month_rollup=nps_dash_acct_month_rollup))

In [ ]:
# spark.sql('''SELECT COUNT(*) FROM {nps_dash_acct_month_rollup_bkp}'''.format(nps_dash_acct_month_rollup_bkp=nps_dash_acct_month_rollup_bkp)).toPandas()

# REGIMEN - HCP IDENTIFICATION

####   1L Regimens with Patient Intensity

In [ ]:
# Step 1: 1L regimens with the patient intensity group (IC eligible or ineligible)
# NOTE: the result is pulled to pandas, so this collects the whole table to the driver
REGIMEN_LOT_TBL = spark.sql(''' 
          select A.*, B.PATIENT_INTENSITY_GROUP
          /* Inferred Description: AML patient line-of-therapy regimen groupings */
          from {lot_grouping_in} A
          left join (
              SELECT * FROM {cohort_in}
          ) B
          ON A.PATIENT_GID = b.PATIENT_GID_RGMN
          '''.format(lot_grouping_in=lot_grouping_in, cohort_in=cohort_in))

REGIMEN_LOT_TBL.createOrReplaceTempView('REGIMEN_LOT_TBL')

REGIMEN_LOT = REGIMEN_LOT_TBL.toPandas()


In [ ]:
print(REGIMEN_LOT)

####   Backbone Selection by Product Hierarchy

In [ ]:
# Define hierarchies
# Step 2: Backbone = the highest ranked product in the regimen, per intensity group
# USER INPUT REQUIRED: both product hierarchies below define the backbone choice
ic_elig_hierarchy = [
    "S-DAC","HI-DAC","VANFLYTA","VYXEOS","CYCLOPHOSPHAMIDE","VINCRISTINE","FLUDARABINE",
    "IDARUBICIN","DAUNORUBICIN","MITOXANTRONE","RYDAPT","VENCLEXTA","XOSPATA","TIBSOVO",
    "REZLIDHIA","IDHIFA","MYLOTARG","DAURISMO","ONUREG","INQOVI","NEXAVAR","ETOPOSIDE",
    "CLADRIBINE","CLOFARABINE","AZACITIDINE","DECITABINE","L-DAC","ARSENIC_TRIOXIDE"
]

ic_inelig_hierarchy = [
    "VENCLEXTA","TIBSOVO","REZLIDHIA","IDHIFA","XOSPATA","DAURISMO","L-DAC",
    "AZACITIDINE","DECITABINE","INQOVI","ONUREG"
]

# Convert hierarchies to dicts for fast lookup
ic_elig_rank = {drug: rank for rank, drug in enumerate(ic_elig_hierarchy, start=1)}
ic_inelig_rank = {drug: rank for rank, drug in enumerate(ic_inelig_hierarchy, start=1)}

# Products absent from the hierarchy fall through to 'UNKNOWN'
def get_backbone(row):
    products = [p.strip() for p in row["LOT_REGIMEN_GROUP"].split(",")]
    if row["PATIENT_INTENSITY_GROUP"] == "IC_ELIG":
        ranks = {p: ic_elig_rank[p] for p in products if p in ic_elig_rank}
    else:  # IC_INELIG
        ranks = {p: ic_inelig_rank[p] for p in products if p in ic_inelig_rank}
    if not ranks:  # no product found in hierarchy
        return "UNKNOWN"
    return min(ranks, key=ranks.get)

# Apply function
REGIMEN_LOT["BACKBONE"] = REGIMEN_LOT.apply(get_backbone, axis=1)

print(REGIMEN_LOT)

In [ ]:
LOT_TBL = spark.createDataFrame(REGIMEN_LOT)
LOT_TBL.createOrReplaceTempView("LOT_TBL")

In [ ]:
LOT_TBL.show()

In [ ]:
# spark.sql('''
#           select * from LOT_TBL
#           ''').toPandas().to_excel('2026-07-24_LOT.xlsx')

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {reg_npi_aci}'''.format(reg_npi_aci=reg_npi_aci))

In [ ]:
# Step 3: Attribute each 1L regimen to the HCP on the first backbone product claim
# IDENTIFYING THE FIRST HCP FOR BACKBONE PRODUCT WITHIN EACH REGIMEN

spark.sql('''
CREATE TABLE {reg_npi_aci}
(
	SELECT B.*, 
	C.ABBOTT_CUSTOMER_ID AS ABBOTT_CUSTOMER_ID_REGIMEN 
	FROM
	(
		SELECT *,
		MAX(NPI_INT) OVER (PARTITION BY PATIENT_GID, REG_NUM) AS NPI_REGIMEN
		FROM
		(
			SELECT *,
			CASE 
				WHEN RNO_BKBONE = 1 THEN NPI_NUMBER 
				ELSE NULL 
			END AS NPI_INT
			FROM
			(
				SELECT *,
				CASE 
					WHEN FINAL_PRODuct_NAME_updated = BACKBONE THEN RNO_PROD --FIRST CLAIM FOR BACKBONE PRODUCT
					ELSE 0 
				END AS RNO_BKBONE
				FROM
				(
					SELECT *,
					ROW_NUMBER() OVER (PARTITION BY PATIENT_GID, REG_NUM, FINAL_PROD_NAME ORDER BY CLAIM_DATE ASC, NPI_NUMBER DESC, CLAIM_ID) AS RNO_PROD,
					ROW_NUMBER() OVER (PARTITION BY PATIENT_GID, REG_NUM ORDER BY CLAIM_DATE ASC, NPI_NUMBER DESC, CLAIM_ID) AS RNO_DATE --FIRST CLAIM BY DATE WITHIN REGIMEN

					FROM
					( 
						SELECT *,
						-- Hypomethylating agents are grouped as HMA
						CASE 
							WHEN FINAL_PRODUCT_NAME_UPDATED IN ('AZACITIDINE','DECITABINE', 'ONUREG', 'INQOVI') THEN 'HMA' 
							ELSE FINAL_PRODUCT_NAME_UPDATED 
						END AS FINAL_PROD_NAME
						FROM
						(
							SELECT A.*,
							B.CLAIM_DATE, B.CLAIM_ID, B.FINAL_PRODUCT_NAME_UPDATED, B.NPI_NUMBER
							FROM
							(
								--SELECT PATIENT_GID, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, BACKBONE, REG_NUM, LOT
								--FROM Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_AML_PATS_COMB_LOT_TBL_BUSINESS_RULE_CHANGE_VAL
								SELECT PATIENT_GID, LOT_REGIMEN_GROUP AS REGIMEN, LOT_START_DATE AS REGIMEN_START_DATE, 
       							 LOT_END_DATE AS REGIMEN_END_DATE, BACKBONE, LOT AS REG_NUM, LOT
								FROM LOT_TBL
								--WHERE LOT = 1
							)A
							LEFT JOIN 
							(
								SELECT DISTINCT PATIENT_SK AS PATIENT_GID, CLAIM_ID, CLAIM_DATE, 
								PROCEDURE_CODE AS PRODUCT_CODE, MASTERED_PRODUCT_NAME AS FINAL_PRODUCT_NAME_UPDATED, MDM_NPI_NUMBER AS NPI_NUMBER
								-- tO BE CONFIRMED
								--FROM {updated_tx_tbl}
								FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
				
								where source_market  =  'ONC'
								and source_flag  =  'SHA_PTD'
								and market_code = 'VENC_AML'
								-- Filgrastim is supportive care, not a treatment product
								AND MASTERED_PRODUCT_NAME <> "FILGRASTIM"
								AND MDM_NPI_NUMBER IS NOT NULL
							)B
							ON A.PATIENT_GID = B.PATIENT_GID
							AND A.BACKBONE = B.FINAL_PRODUCT_NAME_UPDATED
							AND TO_DATE(B.CLAIM_DATE) >= TO_DATE(A.REGIMEN_START_DATE)
							AND TO_DATE(B.CLAIM_DATE) <= TO_DATE(A.REGIMEN_END_DATE)
						)A
					)A 
				)A 
			)A
		)A
	)B
	LEFT JOIN 
	(
        SELECT DISTINCT ABBOTT_CUSTOMER_ID, NPI_NUMBER
        FROM ABV_DDS_SYND.CUSTOMER_TBL
        WHERE DDS_ACTIVE_FLAG = 'Y'  
	)C
	ON B.NPI_REGIMEN = C.NPI_NUMBER
)
'''.format(updated_tx_tbl=updated_tx_tbl,reg_npi_aci=reg_npi_aci))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {reg_npi_aci}
WHERE REG_NUM = 1 AND NPI_REGIMEN IS NULL
'''.format(reg_npi_aci=reg_npi_aci)).show()

# NPS - HEME ACCOUNT TYPE

### ACCOUNT SUBTYPE

####   Account Type Reference

In [ ]:
# Step 4: One account subtype per HCP - most specialised type wins
heme_abbott_account = spark.sql('''
SELECT DISTINCT ABBOTT_CUSTOMER_ID, FINAL_ACCOUNT_TYPE_2, FINAL_ACCOUNT_TYPE_2_HIER
FROM 
(
	SELECT *,
	MIN(FINAL_ACCOUNT_TYPE_2_HIER) OVER (PARTITION BY ABBOTT_CUSTOMER_ID) AS FIN_AC_TYPE
	FROM
	(	
		SELECT *,
		CASE 
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Elite Oncology Center'  THEN 1 --CASE CHECK
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Academic Teaching Hospital' THEN 2
			WHEN FINAL_ACCOUNT_TYPE_2 = 'IDN - Academic Affiliated' THEN 3
			WHEN FINAL_ACCOUNT_TYPE_2 = 'IDN - Community Affiliated' THEN 4
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Corporate' THEN 5
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Other Community' THEN 6
			WHEN FINAL_ACCOUNT_TYPE_2 = 'VA'THEN 7
			WHEN FINAL_ACCOUNT_TYPE_2 = 'DOD'THEN 8
			WHEN FINAL_ACCOUNT_TYPE_2 is NULL THEN 9
		END AS FINAL_ACCOUNT_TYPE_2_HIER
		FROM
		(	
			SELECT A.ABBOTT_CUSTOMER_ID, A.CHILD_ABBOTT_ACCOUNT_ID, 
			TRIM(B.FINAL_ACCOUNT_TYPE_) AS FINAL_ACCOUNT_TYPE_2
			FROM
			(
				SELECT ABBOTT_CUSTOMER_ID, CHILD_ABBOTT_ACCOUNT_ID
				FROM ABV_ADS_SYND.DIM_ACCOUNT_AFFILIATIONS_TBL --UPDATED
				WHERE UNIVERSE_NAME='1VIEW_ONCOLOGY'
				AND AFFILIATION_TYPE = 'HCI-HCP'
                AND SALES_FORCE_CODE = 'ONH2'
				AND DDS_ACTIVE_FLAG = 'Y'
				AND ABBOTT_CUSTOMER_ID IN
				(
					SELECT DISTINCT ABBOTT_CUSTOMER_ID_REGIMEN 
					FROM {reg_npi_aci}
				)

				UNION

				SELECT ABBOTT_CUSTOMER_ID, ORGANIZATION_ABBOTT_ACCOUNT_ID
				FROM {reltio_tbl}  
                WHERE ADS_ACTIVE_FLAG = 'Y'
				AND VALUATION_INCLUDE_FLAG = 'Y'
				AND ABBOTT_CUSTOMER_ID IN 
				(
					SELECT DISTINCT ABBOTT_CUSTOMER_ID_REGIMEN
					FROM {reg_npi_aci}
				)
				AND ABBOTT_CUSTOMER_ID NOT IN
				(
					SELECT DISTINCT ABBOTT_CUSTOMER_ID
					FROM ABV_ADS_SYND.DIM_ACCOUNT_AFFILIATIONS_TBL
					WHERE UNIVERSE_NAME='1VIEW_ONCOLOGY'
					AND AFFILIATION_TYPE = 'HCI-HCP'
                    AND SALES_FORCE_CODE = 'ONH2'
					AND DDS_ACTIVE_FLAG = 'Y'
				) 
			)A
			LEFT JOIN 
			(
				SELECT * 
				FROM {heme_account}
			)B
			ON CAST(A.CHILD_ABBOTT_ACCOUNT_ID AS BIGINT)=CAST(B.CHILD_ACCOUNT_ID AS BIGINT)
		)C
	)C
)C 
WHERE FIN_AC_TYPE = FINAL_ACCOUNT_TYPE_2_HIER
'''.format(reg_npi_aci=reg_npi_aci,heme_account=heme_account,reltio_tbl=reltio_tbl))

heme_abbott_account.createOrReplaceTempView('heme_abbott_account')

In [ ]:
spark.sql('''
SELECT FINAL_ACCOUNT_TYPE_2, FINAL_ACCOUNT_TYPE_2_HIER, COUNT(DISTINCT ABBOTT_CUSTOMER_ID) AS HCP_CT
FROM heme_abbott_account
GROUP BY 1,2
ORDER BY 2
'''.format(reg_npi_aci=reg_npi_aci,heme_account=heme_account,reltio_tbl=reltio_tbl)).toPandas()

### ACCOUNT SUBTYPE GROUP

In [ ]:
# Account grouping: Academic, Academic Satellite, Community split by decile, Federal
heme_abbott_account_group = spark.sql('''
SELECT DISTINCT ABBOTT_CUSTOMER_ID, CHILD_ABBOTT_ACCOUNT_ID, FINAL_ACCOUNT_TYPE_2, FINAL_ACCOUNT_TYPE_2_HIER
FROM 
(
	SELECT *,
	MIN(FINAL_ACCOUNT_TYPE_2_HIER) OVER (PARTITION BY ABBOTT_CUSTOMER_ID) AS FIN_AC_TYPE
	FROM
	(	
		SELECT *,
		CASE 
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Elite Oncology Center'  THEN 1 --CASE CHECK
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Academic Teaching Hospital' THEN 2
			WHEN FINAL_ACCOUNT_TYPE_2 = 'IDN - Academic Affiliated' THEN 3
			WHEN FINAL_ACCOUNT_TYPE_2 = 'IDN - Community Affiliated' THEN 4
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Corporate' THEN 5
			WHEN FINAL_ACCOUNT_TYPE_2 = 'Other Community' THEN 6
			WHEN FINAL_ACCOUNT_TYPE_2 = 'VA'THEN 7
			WHEN FINAL_ACCOUNT_TYPE_2 = 'DOD'THEN 8
			WHEN FINAL_ACCOUNT_TYPE_2 is NULL THEN 9
		END AS FINAL_ACCOUNT_TYPE_2_HIER
		FROM
		(	
			SELECT A.ABBOTT_CUSTOMER_ID, A.CHILD_ABBOTT_ACCOUNT_ID, 
			TRIM(B.FINAL_ACCOUNT_TYPE_) AS FINAL_ACCOUNT_TYPE_2
			FROM
			(
				SELECT ABBOTT_CUSTOMER_ID, CHILD_ABBOTT_ACCOUNT_ID
				FROM ABV_ADS_SYND.DIM_ACCOUNT_AFFILIATIONS_TBL --UPDATED
				WHERE UNIVERSE_NAME='1VIEW_ONCOLOGY'
				AND AFFILIATION_TYPE = 'HCI-HCP'
                AND SALES_FORCE_CODE = 'ONH2'
                AND DDS_ACTIVE_FLAG = 'Y'
				AND ABBOTT_CUSTOMER_ID IN
				(
					SELECT DISTINCT ABBOTT_CUSTOMER_ID_REGIMEN 
					FROM {reg_npi_aci}
				)

				UNION

				SELECT ABBOTT_CUSTOMER_ID, ORGANIZATION_ABBOTT_ACCOUNT_ID
				FROM {reltio_tbl}  
				WHERE ADS_ACTIVE_FLAG = 'Y'
				AND VALUATION_INCLUDE_FLAG = 'Y'
                AND ABBOTT_CUSTOMER_ID IN 
				(
					SELECT DISTINCT ABBOTT_CUSTOMER_ID_REGIMEN
					FROM {reg_npi_aci}
				)
				AND ABBOTT_CUSTOMER_ID NOT IN
				(
					SELECT DISTINCT ABBOTT_CUSTOMER_ID
					FROM ABV_ADS_SYND.DIM_ACCOUNT_AFFILIATIONS_TBL
					WHERE UNIVERSE_NAME='1VIEW_ONCOLOGY'
					AND AFFILIATION_TYPE = 'HCI-HCP'
                    AND SALES_FORCE_CODE = 'ONH2'
               		AND DDS_ACTIVE_FLAG = 'Y'
				) 
			)A
			LEFT JOIN 
			(
				SELECT * 
				FROM {heme_account}
			)B
			ON CAST(A.CHILD_ABBOTT_ACCOUNT_ID AS BIGINT)=CAST(B.CHILD_ACCOUNT_ID AS BIGINT)
		)C
	)C
)C 
WHERE FIN_AC_TYPE = FINAL_ACCOUNT_TYPE_2_HIER
'''.format(reg_npi_aci=reg_npi_aci,heme_account=heme_account,reltio_tbl=reltio_tbl))

heme_abbott_account_group.createOrReplaceTempView('heme_abbott_account_group')

In [ ]:
acct_type_grp_tbl

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {acct_type_grp_tbl}'''.format(acct_type_grp_tbl=acct_type_grp_tbl))

In [ ]:
spark.sql('''
CREATE TABLE {acct_type_grp_tbl} AS
(
    SELECT DISTINCT ABBOTT_CUSTOMER_ID, FINAL_ACCOUNT_TYPE_GROUP_2,
    CASE
        WHEN FINAL_ACCOUNT_TYPE_GROUP_2 = 'Academic' THEN 1
        WHEN FINAL_ACCOUNT_TYPE_GROUP_2 = 'Academic Satellite' THEN 2
        WHEN FINAL_ACCOUNT_TYPE_GROUP_2 = 'Larger Community' THEN 3
        WHEN FINAL_ACCOUNT_TYPE_GROUP_2 = 'Smaller Community' THEN 4
        WHEN FINAL_ACCOUNT_TYPE_GROUP_2 = 'Federal' THEN 5
        WHEN FINAL_ACCOUNT_TYPE_GROUP_2 = 'Not Available' THEN 6
    END AS FINAL_ACCOUNT_TYPE_GROUP_2_HIER
    FROM
    (
        SELECT *,
        CASE
            WHEN MAX_ACCOUNT_DECILE IS NULL AND FINAL_ACCOUNT_TYPE_GROUP = 'Community' THEN 'Smaller Community'
            WHEN MAX_ACCOUNT_DECILE IS NOT NULL AND FINAL_ACCOUNT_TYPE_GROUP = 'Community' THEN
                CASE
                    WHEN MAX_ACCOUNT_DECILE >= 8 THEN 'Larger Community'
                    ELSE 'Smaller Community'
                END 
            WHEN FINAL_ACCOUNT_TYPE_GROUP IS NULL THEN 'Not Available'
            ELSE FINAL_ACCOUNT_TYPE_GROUP
        END AS FINAL_ACCOUNT_TYPE_GROUP_2
        FROM
        (
            SELECT *,
            MAX(ACCOUNT_DECILE) OVER (PARTITION BY ABBOTT_CUSTOMER_ID) AS MAX_ACCOUNT_DECILE
            FROM
            (
                SELECT A.*, B.DECILE AS ACCOUNT_DECILE
                FROM
                (
                    SELECT *,
                    CASE
                        WHEN FINAL_ACCOUNT_TYPE_2 IN ('Elite Oncology Center','Academic Teaching Hospital') THEN 'Academic'
                        WHEN FINAL_ACCOUNT_TYPE_2 IN ('IDN - Academic Affiliated') THEN 'Academic Satellite'
                        WHEN FINAL_ACCOUNT_TYPE_2 IN ('IDN - Community Affiliated','Corporate','Other Community') THEN 'Community'
                        WHEN FINAL_ACCOUNT_TYPE_2 IN ('VA','DOD') THEN 'Federal'
                        ELSE NULL
                    END AS FINAL_ACCOUNT_TYPE_GROUP
                    FROM heme_abbott_account_group
                )A
                LEFT JOIN
                (
                    SELECT *
                    FROM Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_AML_CHILD_ACCOUNT_DECILE
                )B
                ON CAST(A.CHILD_ABBOTT_ACCOUNT_ID AS BIGINT)=CAST(B.ABBOTT_ACCOUNT_ID AS BIGINT)
            )A
        )A
    )A
)
'''.format(acct_type_grp_tbl=acct_type_grp_tbl))

In [ ]:
spark.sql('''
SELECT COUNT(*)
FROM {acct_type_grp_tbl}
'''.format(acct_type_grp_tbl=acct_type_grp_tbl)).toPandas()

In [ ]:
spark.sql('''
SELECT DISTINCT FINAL_ACCOUNT_TYPE_GROUP_2, FINAL_ACCOUNT_TYPE_GROUP_2_HIER
FROM {acct_type_grp_tbl}
ORDER BY 2
'''.format(acct_type_grp_tbl=acct_type_grp_tbl)).toPandas()

# NPS DASH - HCP INFO MAP

In [ ]:
npi_hcp_seg_map

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {npi_hcp_seg_map}'''.format(npi_hcp_seg_map=npi_hcp_seg_map))

In [ ]:
# Step 5: Attach HCP decile, above-brand and execution segments to each regimen
spark.sql('''
CREATE TABLE {npi_hcp_seg_map}
(
	SELECT A.*, 
    B.UNI_DECILE, B.ABOVE_BRAND_SEGMENT,
    C.AML_EXECUTION_SEGMENT AS EXECUTION_SEGMENT,
    D.FINAL_ACCOUNT_TYPE_2
	FROM 
	(
		SELECT DISTINCT PATIENT_GID, REG_NUM, LOT, REGIMEN_START_DATE, REGIMEN_END_DATE, 
        REGIMEN, BACKBONE, NPI_REGIMEN, ABBOTT_CUSTOMER_ID_REGIMEN
		FROM {reg_npi_aci}
	)A
	LEFT JOIN
	(
		SELECT * 
		FROM {above_brand}
	)B
	ON A.NPI_REGIMEN=B.NPI
	LEFT JOIN
	(
		SELECT * 
		FROM {gne_brand}
	)C
	ON A.NPI_REGIMEN=C.NPI
	LEFT JOIN
	(
		SELECT *
		FROM heme_abbott_account
	)D
	ON A.ABBOTT_CUSTOMER_ID_REGIMEN=D.ABBOTT_CUSTOMER_ID
)'''.format(above_brand=above_brand,gne_brand=gne_brand,npi_hcp_seg_map=npi_hcp_seg_map,reg_npi_aci=reg_npi_aci))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {npi_hcp_seg_map}
WHERE FINAL_ACCOUNT_TYPE_2 IS NOT NULL
'''.format(npi_hcp_seg_map=npi_hcp_seg_map)).show()

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT ABBOTT_CUSTOMER_ID_REGIMEN)
FROM {npi_hcp_seg_map}
WHERE FINAL_ACCOUNT_TYPE_2 IS NOT NULL
'''.format(npi_hcp_seg_map=npi_hcp_seg_map)).show()

# NPS DASH - FINAL TABLE

#### FINAL TABLE

In [ ]:
elig_flags_final

In [ ]:
# spark.sql('''
# SELECT COUNT(DISTINCT PATIENT_GID)
# FROM {elig_flags_final}
# WHERE DX_LB_ELIG_FLG = 1 
# AND TX_MX_LB_ELIG_FLG = 1 AND TX_RX_LB_ELIG_FLG = 1
# AND DX_TX_DIFF_FLG = 1 AND SCT_DX_PX_FLG = 0 AND ARSENIC_FLG = 1 
# AND YEAR(FIRST_DX) >= 2019
# AND PATIENT_COHORT = 'IC_INELIG'
# '''.format(elig_flags_final=elig_flags_final)).show()

####   Dashboard Table - Backbone Grouping and Null Handling

In [ ]:
npi_hcp_seg_map

In [ ]:
# Step 6: Backbone grouped as HMA, Venclexta or other novel agents; nulls made explicit
dash_tbl = spark.sql('''
SELECT *,
CASE 
    -- BACKBONE carries drug names, so HMA agents are listed explicitly
    WHEN BACKBONE = 'VENCLEXTA' THEN 'VENCLEXTA'
    WHEN BACKBONE IN ('AZACITIDINE','DECITABINE','ONUREG','INQOVI') THEN 'HMA'
    ELSE 'OTHER NOVEL AGENTS' --MIGHT INCLUDE SOME NON-NOVEL AGENTS AS WELL
END AS BACKBONE_GROUP,
CASE WHEN NPI_REGIMEN IS NULL THEN "Not Available" ELSE NPI_REGIMEN END AS NPI_REGIMEN_2,
CASE WHEN ABBOTT_CUSTOMER_ID_REGIMEN IS NULL THEN "Not Available" ELSE ABBOTT_CUSTOMER_ID_REGIMEN END AS ABBOTT_CUSTOMER_ID_REGIMEN_2,
CASE WHEN (UNI_DECILE IS NULL OR UNI_DECILE="-") THEN "Not Available" ELSE UNI_DECILE END AS UNI_DECILE_2,
CASE WHEN ABOVE_BRAND_SEGMENT IS NULL THEN "Not Available" ELSE ABOVE_BRAND_SEGMENT END AS ABOVE_BRAND_SEGMENT_2,
CASE WHEN EXECUTION_SEGMENT IS NULL THEN "Not Available" ELSE EXECUTION_SEGMENT END AS EXECUTION_SEGMENT_2,
CASE WHEN FINAL_ACCOUNT_TYPE_2 IS NULL THEN "Not Available" ELSE FINAL_ACCOUNT_TYPE_2 END AS FINAL_ACCOUNT_TYPE_2_UPD
FROM {npi_hcp_seg_map}
-- All lines of therapy retained; do not reinstate the 1L filter below
--WHERE REG_NUM = 1
--AND PATIENT_GID IN
--(
   -- SELECT DISTINCT PATIENT_GID
   -- FROM elig_flags_final
   -- WHERE DX_LB_ELIG_FLG = 1 
    --AND TX_MX_LB_ELIG_FLG = 1 AND TX_RX_LB_ELIG_FLG = 1
    --AND DX_TX_DIFF_FLG = 1 AND SCT_DX_PX_FLG = 0 AND ARSENIC_FLG = 1 
    --AND YEAR(FIRST_DX) >= 2019
    --AND PATIENT_COHORT = 'IC_INELIG'
--)
'''.format(npi_hcp_seg_map=npi_hcp_seg_map,elig_flags_final=elig_flags_final))

dash_tbl.createOrReplaceTempView('dash_tbl')

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM dash_tbl
'''.format(npi_hcp_seg_map=npi_hcp_seg_map,elig_flags_final=elig_flags_final)).show()

#### PBI INPUT TABLE

In [ ]:
dash_tbl_final

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {dash_tbl_final}'''.format(dash_tbl_final=dash_tbl_final))

In [ ]:
spark.sql('''
-- Step 7: Power BI dataset - one row per patient and line of therapy
CREATE TABLE {dash_tbl_final} AS
(
    SELECT *,
    CASE 
        WHEN ACCT_SUBTYPE = 'Elite Oncology Center'  THEN 1 --CASE CHECK
        WHEN ACCT_SUBTYPE = 'Academic Teaching Hospital' THEN 2
        WHEN ACCT_SUBTYPE = 'IDN - Academic Affiliated' THEN 3
        WHEN ACCT_SUBTYPE = 'IDN - Community Affiliated' THEN 4
        WHEN ACCT_SUBTYPE = 'Corporate' THEN 5
        WHEN ACCT_SUBTYPE = 'Other Community' THEN 6
        WHEN ACCT_SUBTYPE = 'VA'THEN 7
        WHEN ACCT_SUBTYPE = 'DOD'THEN 8
        ELSE 9
    END AS ACCT_SUBTYPE_HIER
    FROM
    (
        -- LOT keeps the lot grain; without it DISTINCT would collapse lines of therapy
        SELECT DISTINCT PATIENT_GID, LOT, REGIMEN, BACKBONE, BACKBONE_GROUP,
        REGIMEN_START_DATE, REGIMEN_END_DATE, 
        UNI_DECILE_2 AS UNI_DECILE,
        NPI_REGIMEN_2 AS NPI, ABBOTT_CUSTOMER_ID_REGIMEN_2 AS ABBOTT_ID,
        ABOVE_BRAND_SEGMENT_2 AS ABOVE_BRAND_SEGMENT, EXECUTION_SEGMENT_2 AS GNE_SEGMENT, 
        FINAL_ACCOUNT_TYPE_2_UPD AS ACCT_SUBTYPE
        FROM dash_tbl
    )A
)
'''.format(dash_tbl_final=dash_tbl_final))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {dash_tbl_final}
'''.format(dash_tbl_final=dash_tbl_final)).show()

In [ ]:
spark.sql('''
select DISTINCT ACCT_SUBTYPE, ACCT_SUBTYPE_HIER
from {dash_tbl_final} 
ORDER BY 2 
'''.format(dash_tbl=dash_tbl,dash_tbl_final=dash_tbl_final)).toPandas()

In [ ]:
elig_flag_df = spark.sql(f"""
    select distinct patient_gid, dx_lb_elig_flg, dx_tx_diff_flg, patient_cohort 
    from {flags_in}          
""")
elig_flag_df.createOrReplaceTempView('elig_flag_df')

In [ ]:
dash_tbl_final

In [ ]:
spark.sql('''
SELECT *
FROM {dash_tbl_final}
'''.format(dash_tbl_final=dash_tbl_final)).show()

In [ ]:
# 'month' is shadowed by the reporting-month global, so alias the Spark functions
from pyspark.sql.functions import (concat, lit, lpad, coalesce, when,
                                   month as f_month, year as f_year)


In [ ]:
dash_tbl_df = spark.sql('''
    SELECT *
    FROM {dash_tbl_final}
'''.format(dash_tbl_final=dash_tbl_final))

# Calendar parts for reporting: year, month number, sortable YYYY-MM label,
# plus the semester label the published Summary sheet was built on
def add_date_parts(df, date_col, prefix):
    return (
        df
        .withColumn(prefix + '_year', f_year(col(date_col)))
        .withColumn(prefix + '_month', f_month(col(date_col)))
        .withColumn(prefix + '_year_month', concat(f_year(col(date_col)), lit('-'),
                                                   lpad(f_month(col(date_col)).cast('string'), 2, '0')))
        .withColumn(prefix + '_sem', concat(f_year(col(date_col)), lit(' - S'),
                                            ((f_month(col(date_col)) + 5) / 6).cast('int')))
    )

# Eligibility flags and patient cohort are attached at patient grain.
# Not every treated patient reaches the flags table, so the join is a LEFT join.
dash_tbl_df = dash_tbl_df.join(elig_flag_df, how='left', on=['patient_gid'])

# Unmatched patients are labelled rather than left null: a null would become an
# unlabelled group in the summary and read as a blank row in Excel
dash_tbl_df = (
    dash_tbl_df
    .withColumn('flags_matched', when(col('patient_cohort').isNull(), lit(0)).otherwise(lit(1)))
    .withColumn('dx_lb_elig_flg', coalesce(col('dx_lb_elig_flg'), lit(-1)))
    .withColumn('dx_tx_diff_flg', coalesce(col('dx_tx_diff_flg'), lit(-1)))
    .withColumn('patient_cohort', coalesce(col('patient_cohort'), lit('Not Available')))
)

dash_tbl_df = add_date_parts(dash_tbl_df, 'regimen_start_date', 'regimen_start')
dash_tbl_df = add_date_parts(dash_tbl_df, 'regimen_end_date', 'regimen_end')

# Patients with no eligibility record; -1 and 'Not Available' mark them in the summary
print('rows with no eligibility match:', dash_tbl_df.filter(col('flags_matched') == 0).count())


In [ ]:
dash_tbl_df.count()

In [ ]:
# USER INPUT REQUIRED: extract file name
# Column order mirrors the AML NPS sheet, plus LOT and the month level columns
EXTRACT_COLS = [
    'PATIENT_GID', 'LOT',
    'DX_LB_ELIG_FLG', 'DX_TX_DIFF_FLG', 'PATIENT_COHORT', 'FLAGS_MATCHED',
    'REGIMEN', 'BACKBONE', 'BACKBONE_GROUP',
    'REGIMEN_START_DATE', 'REGIMEN_START_YEAR', 'REGIMEN_START_MONTH',
    'REGIMEN_START_YEAR_MONTH', 'REGIMEN_START_SEM',
    'REGIMEN_END_DATE', 'REGIMEN_END_YEAR', 'REGIMEN_END_MONTH',
    'REGIMEN_END_YEAR_MONTH', 'REGIMEN_END_SEM',
    'NPI', 'ABBOTT_ID', 'UNI_DECILE',
    'ABOVE_BRAND_SEGMENT', 'GNE_SEGMENT', 'ACCT_SUBTYPE', 'ACCT_SUBTYPE_HIER',
]

extract_df = dash_tbl_df.select(*EXTRACT_COLS)
# extract_df.toPandas().to_excel('2026-07-24_aml_nps_dash_LEGACY_EXTRACT.xlsx', index=False)


# NPS SUMMARY BY LOT, YEAR AND MONTH

Rebuilds the two Summary-sheet tables - NPS Volume and NPS Shares - at year and month
level, with LOT added as a grain so 2L, 3L and later lines report alongside 1L.

The eligibility flags and patient cohort are kept as columns rather than filters, so one
table serves every slice the workbook pivots on. The published Summary sheet corresponds
to `DX_LB_ELIG_FLG = 1`, `DX_TX_DIFF_FLG = 1`, `PATIENT_COHORT = 'IC_INELIG'`, `LOT = 1`
rolled up from months to semesters.


####   Summary Inputs and Working View


In [ ]:
# Reporting floor for the summary; set to None to keep every year
SUMMARY_START_YEAR = 2019

# USER INPUT REQUIRED: report on regimen start or regimen end
SUMMARY_TIME_BASIS = 'END'             # 'START' or 'END'

_basis = 'REGIMEN_START' if SUMMARY_TIME_BASIS == 'START' else 'REGIMEN_END'
SUMMARY_YEAR_COL = _basis + '_YEAR'
SUMMARY_MONTH_COL = _basis + '_MONTH'
SUMMARY_YEAR_MONTH_COL = _basis + '_YEAR_MONTH'

# Eligibility flags and patient cohort stay as columns so the summary can be
# sliced any way downstream, exactly like the pivot filters on the workbook
SUMMARY_DIMS = ['DX_LB_ELIG_FLG', 'DX_TX_DIFF_FLG', 'PATIENT_COHORT', 'LOT']

# Year and month are the reporting grain
SUMMARY_TIME_DIMS = ['TIME_YEAR', 'TIME_MONTH', 'TIME_YEAR_MONTH']

# Reporting floor is a window, not a cohort rule, so it stays a filter
SUMMARY_WHERE = ('WHERE {year_col} >= {yr}'.format(year_col=SUMMARY_YEAR_COL, yr=SUMMARY_START_YEAR)
                 if SUMMARY_START_YEAR is not None else '')

# dash_tbl_df carries LOT, the eligibility flags, patient cohort and the calendar parts
dash_tbl_df.createOrReplaceTempView('dash_tbl_lot')


####   Summary Table Build


In [ ]:
spark.sql('''DROP TABLE IF EXISTS {nps_lot_month_summary}'''.format(nps_lot_month_summary=nps_lot_month_summary))


In [ ]:
spark.sql('''
-- Step 10: NPS volume and share by eligibility, cohort, line of therapy, year and month
CREATE TABLE {nps_lot_month_summary} AS
(
    SELECT {dims}, {time_dims},
    SUM(VEN_FLAG) AS VEN_NPS,
    SUM(HMA_FLAG) AS HMA_NPS,
    SUM(OTH_FLAG) AS OTHERS_NPS,
    COUNT(*) AS TOTAL_NPS,
    SUM(VEN_FLAG)/COUNT(*) AS VEN_SHARE,
    SUM(HMA_FLAG)/COUNT(*) AS HMA_SHARE,
    SUM(OTH_FLAG)/COUNT(*) AS OTHERS_SHARE
    FROM
    (
        -- One new patient start per patient and line of therapy
        SELECT DISTINCT PATIENT_GID, {dims},
        {year_col} AS TIME_YEAR,
        {month_col} AS TIME_MONTH,
        {year_month_col} AS TIME_YEAR_MONTH,
        CASE WHEN BACKBONE_GROUP = 'VENCLEXTA' THEN 1 ELSE 0 END AS VEN_FLAG,
        CASE WHEN BACKBONE_GROUP = 'HMA' THEN 1 ELSE 0 END AS HMA_FLAG,
        CASE WHEN BACKBONE_GROUP = 'OTHER NOVEL AGENTS' THEN 1 ELSE 0 END AS OTH_FLAG
        FROM dash_tbl_lot
        {where}
    )A
    GROUP BY {dims}, {time_dims}
)
'''.format(nps_lot_month_summary=nps_lot_month_summary,
           dims=', '.join(SUMMARY_DIMS),
           time_dims=', '.join(SUMMARY_TIME_DIMS),
           year_col=SUMMARY_YEAR_COL,
           month_col=SUMMARY_MONTH_COL,
           year_month_col=SUMMARY_YEAR_MONTH_COL,
           where=SUMMARY_WHERE))


####   Summary Tables - NPS Volume and NPS Shares by Month


In [ ]:
summary_df = spark.sql('''
SELECT {dims}, {time_dims}, VEN_NPS, HMA_NPS, OTHERS_NPS, TOTAL_NPS,
VEN_SHARE, HMA_SHARE, OTHERS_SHARE
FROM {nps_lot_month_summary}
ORDER BY {dims}, TIME_YEAR, TIME_MONTH
'''.format(nps_lot_month_summary=nps_lot_month_summary,
           dims=', '.join(SUMMARY_DIMS),
           time_dims=', '.join(SUMMARY_TIME_DIMS))).toPandas()

_LABELS = {'TIME_YEAR': 'Year', 'TIME_MONTH': 'Month', 'TIME_YEAR_MONTH': 'Year Month',
           'VEN_NPS': 'VEN', 'HMA_NPS': 'HMA', 'OTHERS_NPS': 'Others', 'TOTAL_NPS': 'Total',
           'VEN_SHARE': 'VEN', 'HMA_SHARE': 'HMA', 'OTHERS_SHARE': 'Others'}

# Table 1 of the Summary sheet: new patient start volume
nps_volume = (summary_df[SUMMARY_DIMS + SUMMARY_TIME_DIMS
                         + ['VEN_NPS', 'HMA_NPS', 'OTHERS_NPS', 'TOTAL_NPS']]
              .rename(columns=_LABELS))

# Table 2 of the Summary sheet: backbone share of starts within each month
nps_shares = (summary_df[SUMMARY_DIMS + SUMMARY_TIME_DIMS
                         + ['VEN_SHARE', 'HMA_SHARE', 'OTHERS_SHARE']]
              .rename(columns=_LABELS))

print(nps_volume)
print(nps_shares)


In [ ]:
# USER INPUT REQUIRED: summary export file name
# Filename records which date the reporting month was taken from
with pd.ExcelWriter(f'2026-07-24_aml_nps_summary_by_lot_month_{SUMMARY_TIME_BASIS.lower()}_SATEEK.xlsx') as xl:
    nps_volume.to_excel(xl, sheet_name='NPS Volume', index=False)
    nps_shares.to_excel(xl, sheet_name='NPS Shares', index=False)


####   QC - Month Totals Must Roll Up to the Published Semesters


In [ ]:
# The published Summary sheet slice: IC ineligible with both eligibility flags set
PUBLISHED_SLICE = ((summary_df['DX_LB_ELIG_FLG'] == 1)
                   & (summary_df['DX_TX_DIFF_FLG'] == 1)
                   & (summary_df['PATIENT_COHORT'] == 'IC_INELIG'))

lot1 = summary_df[PUBLISHED_SLICE & (summary_df['LOT'] == 1)]

# Monthly detail for the published slice
print(lot1[['TIME_YEAR_MONTH', 'VEN_NPS', 'HMA_NPS', 'OTHERS_NPS', 'TOTAL_NPS', 'VEN_SHARE']])

# Months roll up to semesters; shares must be recomputed, never averaged
sem_rollup = (lot1.assign(SEM=lot1['TIME_YEAR'].astype(str) + ' - S'
                          + ((lot1['TIME_MONTH'] + 5) // 6).astype(str))
                  .groupby('SEM')[['VEN_NPS', 'HMA_NPS', 'OTHERS_NPS', 'TOTAL_NPS']].sum())
sem_rollup['VEN_SHARE'] = sem_rollup['VEN_NPS'] / sem_rollup['TOTAL_NPS']
print(sem_rollup)

# Volume by line of therapy; expect a steep fall from 1L and clinically plausible depth
print(summary_df[PUBLISHED_SLICE].groupby('LOT')['TOTAL_NPS'].sum().head(15))
print('max LOT present:', summary_df['LOT'].max())

# Patients with no eligibility record land in this bucket, not in a blank row
unmatched = summary_df[summary_df['PATIENT_COHORT'] == 'Not Available']['TOTAL_NPS'].sum()
print('starts with no eligibility match:', unmatched)

# TOTAL_NPS is per slice; summing across slices multiply-counts
print(summary_df.groupby(['PATIENT_COHORT', 'DX_LB_ELIG_FLG', 'DX_TX_DIFF_FLG'])['TOTAL_NPS'].sum())


####   Ad-hoc Extract and Checks

In [ ]:
spark.sql('''
          
          
          sel
          
          
          ''').show()

In [ ]:
spark.stop()

# NPS DASH - INDEXING

### INDEX CREATION

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {ipsos_index_tbl}'''.format(ipsos_index_tbl=ipsos_index_tbl))

In [ ]:
spark.sql('''
-- Step 8: Index the reported NPS share against the share observed in SHA claims
CREATE TABLE {ipsos_index_tbl} AS
(
	SELECT MONTH_REPO, 
	VEN_NPS AS VEN_NPS_REP, TOTAL_NPS AS TOTAL_NPS_REP,
	VEN_NPS_SHA, TOTAL_NPS_SHA
	FROM
	(
		SELECT A.*, B.VEN_NPS_SHA, B.TOTAL_NPS_SHA, B.SHARE_SHA
		FROM
		(
			SELECT TO_DATE(CAST(UNIX_TIMESTAMP(MONTH_REP, 'MM/dd/yyyy') AS TIMESTAMP)) AS MONTH_REPO,
			VEN_NPS, TOTAL_NPS,
			(VEN_NPS/TOTAL_NPS) AS SHARE_REP 
			FROM {ipsos_repo_tbl}
		)A
		LEFT JOIN
		(
			SELECT DISTINCT TO_DATE(CAST(UNIX_TIMESTAMP(REGIMEN_MONTH, 'MM/dd/yyyy') AS TIMESTAMP)) AS REGIMEN_MONTH_SHA,
			VEN_NPS_SHA, TOTAL_NPS_SHA, (VEN_NPS_SHA/TOTAL_NPS_SHA) AS SHARE_SHA
			FROM
			(
				SELECT REGIMEN_MONTH, 
				SUM(VEN_FLAG) AS VEN_NPS_SHA,
				COUNT(VEN_FLAG) AS TOTAL_NPS_SHA
				FROM
				(
					SELECT *, 
					CASE WHEN BACKBONE='VENCLEXTA' THEN 1 ELSE 0 END AS VEN_FLAG
					FROM
					( 
						SELECT *, 
						CASE 
							WHEN MONTH(REGIMEN_START_DATE) > 9 THEN CONCAT(MONTH(REGIMEN_START_DATE),'/01/',YEAR(REGIMEN_START_DATE)) 
							ELSE CONCAT('0', MONTH(REGIMEN_START_DATE),'/01/',YEAR(REGIMEN_START_DATE))
						END AS REGIMEN_MONTH
						FROM 
						(
							SELECT * 
							FROM dash_tbl
						)
					)
				)
				GROUP BY 1
			)
		)B 
		ON A.MONTH_REPO=B.REGIMEN_MONTH_SHA
	)A
	-- USER INPUT REQUIRED: reporting floor, Jan 2019
	WHERE MONTH_REPO >= TO_DATE('2019-01-01')
)
'''.format(ipsos_index_tbl=ipsos_index_tbl,ipsos_repo_tbl=ipsos_repo_tbl))

In [ ]:
spark.sql('''
SELECT *
FROM {ipsos_index_tbl}
'''.format(ipsos_index_tbl=ipsos_index_tbl)).toPandas().to_csv('index_map_Jan26.csv')

### INDEX APPLICATION

In [ ]:
nps_dash_acct_month_rollup

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {nps_dash_acct_month_rollup}'''.format(nps_dash_acct_month_rollup=nps_dash_acct_month_rollup))

In [ ]:
spark.sql('''
-- Step 9: FINAL ROLLUP - NPS counts by month, account subtype and account group
CREATE TABLE {nps_dash_acct_month_rollup} AS
(
	SELECT INDX.*, 
	SHS.ACCT_SUB_TYPE_GRP, SHS.ACCT_SUB_TYPE, 
	SHS.VEN_NPS_SHA, SHS.HMA_NPS_SHA, SHS.NOV_NPS_SHA, SHS.TOTAL_NPS_SHA,
	CASE 
        WHEN ACCT_SUB_TYPE = 'Elite Oncology Center'  THEN 1 --CASE CHECK
        WHEN ACCT_SUB_TYPE = 'Academic Teaching Hospital' THEN 2
        WHEN ACCT_SUB_TYPE = 'IDN - Academic Affiliated' THEN 3
        WHEN ACCT_SUB_TYPE = 'IDN - Community Affiliated' THEN 4
        WHEN ACCT_SUB_TYPE = 'Corporate' THEN 5
        WHEN ACCT_SUB_TYPE = 'Other Community' THEN 6
        WHEN ACCT_SUB_TYPE = 'VA'THEN 7
        WHEN ACCT_SUB_TYPE = 'DOD'THEN 8
        ELSE 9
    END AS ACCT_SUB_TYPE_HIER,
	CASE
		WHEN ACCT_SUB_TYPE_GRP = 'Academic' THEN 1
		WHEN ACCT_SUB_TYPE_GRP = 'Community' THEN 2
	END AS ACCT_SUB_TYPE_GRP_HIER
	FROM
	(
		SELECT DISTINCT MONTH_REPO
		FROM {ipsos_index_tbl}
	)INDX
	LEFT JOIN
	(
		SELECT DISTINCT TO_DATE(CAST(UNIX_TIMESTAMP(REGIMEN_MONTH, 'MM/dd/yyyy') AS TIMESTAMP)) AS REGIMEN_MONTH_SHA,
		ACCT_SUB_TYPE_GRP,
		ACCT_SUB_TYPE,
		SUM(VEN_FLAG) AS VEN_NPS_SHA,
		SUM(HMA_FLAG) AS HMA_NPS_SHA,
		SUM(NOV_FLAG) AS NOV_NPS_SHA,
		COUNT(VEN_FLAG) AS TOTAL_NPS_SHA
		FROM
		(
			SELECT *, 
			CASE 
				WHEN MONTH(REGIMEN_START_DATE) > 9 THEN CONCAT(MONTH(REGIMEN_START_DATE),'/01/',YEAR(REGIMEN_START_DATE)) 
				ELSE CONCAT('0', MONTH(REGIMEN_START_DATE),'/01/',YEAR(REGIMEN_START_DATE))
			END AS REGIMEN_MONTH,
			-- Venclexta, HMA and other novel agent starts counted separately
			CASE WHEN BACKBONE_GROUP='VENCLEXTA' THEN 1 ELSE 0 END AS VEN_FLAG,
			CASE WHEN BACKBONE_GROUP='HMA' THEN 1 ELSE 0 END AS HMA_FLAG,
			CASE WHEN BACKBONE_GROUP='OTHER NOVEL AGENTS' THEN 1 ELSE 0 END AS NOV_FLAG,
			CASE
				WHEN ACCT_SUB_TYPE IN ('Elite Oncology Center', 'Academic Teaching Hospital') THEN 'Academic'
				ELSE 'Community'
			END AS ACCT_SUB_TYPE_GRP
			FROM 
			(
				SELECT DISTINCT PATIENT_GID, BACKBONE, BACKBONE_GROUP, REGIMEN_START_DATE, FINAL_ACCOUNT_TYPE_2_UPD AS ACCT_SUB_TYPE
				FROM dash_tbl
			)A
		)A
		GROUP BY 1,2,3
	)SHS
	ON INDX.MONTH_REPO = SHS.REGIMEN_MONTH_SHA
)
'''.format(nps_dash_acct_month_rollup=nps_dash_acct_month_rollup,ipsos_index_tbl=ipsos_index_tbl))

In [ ]:
spark.sql('''
SELECT *
FROM {nps_dash_acct_month_rollup}
'''.format(nps_dash_acct_month_rollup=nps_dash_acct_month_rollup)).toPandas().to_csv('index_map_acct_Jan26.csv')

In [ ]:
spark.sql('''
SELECT SUM(VEN_NPS_SHA) AS VEN_SHA, 
SUM(HMA_NPS_SHA) AS HMA_SHA, 
SUM(NOV_NPS_SHA) AS NOV_SHA, 
SUM(TOTAL_NPS_SHA) AS TOT_SHA
FROM {nps_dash_acct_month_rollup}
'''.format(nps_dash_acct_month_rollup=nps_dash_acct_month_rollup)).toPandas()

In [ ]:
spark.sql('''
SELECT DISTINCT ACCT_SUB_TYPE, ACCT_SUB_TYPE_HIER
FROM {nps_dash_acct_month_rollup}
ORDER BY 2
'''.format(nps_dash_acct_month_rollup=nps_dash_acct_month_rollup)).toPandas()

In [ ]:
spark.sql('''
SELECT DISTINCT ACCT_SUB_TYPE_GRP, ACCT_SUB_TYPE_GRP_HIER
FROM {nps_dash_acct_month_rollup}
ORDER BY 2
'''.format(nps_dash_acct_month_rollup=nps_dash_acct_month_rollup)).toPandas()

In [ ]:
spark.stop()